In [7]:
%%capture
"""
install the upgraded givernylocal library.
    - n.b. this only needs to be run once.
"""
!pip install --upgrade givernylocal

<hr style = "height:6px;border:none;background-color:sienna">

<h2 style = "font-weight:bold;font-style:italic">
    Getdata demo notebook
</h2>

<p style = "font-weight:bold;font-size:13px">
    &emsp;n.b. requires python 3.9+
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>purpose</u> :
    <br>
    &emsp;- local processing of JHTDB datasets.
    <br><br>
    <u>supported datasets</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        isotropic1024coarse &ensp;: &ensp;isotropic 1024-cube (coarse).
    </li>
    <li style = "font-weight:bold;font-size:13px">
        isotropic1024fine &ensp;: &ensp;isotropic 1024-cube (fine).
    </li>
    <li style = "font-weight:bold;font-size:13px">
        isotropic4096 &ensp;: &ensp;isotropic 4096-cube.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        isotropic8192 &ensp;: &ensp;isotropic 8192-cube.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        isotropic32768 &ensp;: &ensp;isotropic 32768-cube.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        sabl2048low &ensp;: &ensp;stable atmospheric boundary layer 2048-cube, low-rate timestep.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        sabl2048high &ensp;: &ensp;stable atmospheric boundary layer 2048-cube, high-rate timestep.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        stsabl2048low &ensp;: &ensp;strong stable atmospheric boundary layer 2048-cube, low-rate timestep.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        stsabl2048high &ensp;: &ensp;strong stable atmospheric boundary layer 2048-cube, high-rate timestep.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        rotstrat4096 &ensp;: &ensp;rotating stratified 4096-cube.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        mhd1024 &ensp;: &ensp;magneto-hydrodynamic isotropic 1024-cube.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        mixing &ensp;: &ensp;homogeneous buoyancy driven 1024-cube.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        channel &ensp;: &ensp;channel flow.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        channel5200 &ensp;: &ensp;channel flow (reynolds number 5200).
    </li>
    <li style = "font-weight:bold;font-size:13px">
        transition_bl &ensp;: &ensp;transitional boundary layer.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        diurnal_windfarm &ensp;: &ensp;diurnal windfarm.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        nbl_windfarm &ensp;: &ensp;neutral boundary layer windfarm.
    </li>
</ul>

<p style = "font-weight:bold;font-size:13px">
    <u>functions</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        getData &ensp;: &ensp;retrieve (interpolate and/or differentiate) field data on a set of specified spatial points for the specified variable.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        write_interpolation_tsv_file &ensp;: &ensp;write getData results to a .tsv file.
    </li>
</ul>

<hr style = "height:6px;border:none;background-color:sienna">

<h4 style = "font-weight:bold;font-style:italic">
    instantiate dataset
</h4>

<p style = "font-weight:bold;font-size:13px">
    <u>purpose</u> : 
    <br>
    &emsp;- instantiate the dataset and cache the metadata.
    <br>

</p>

<p style = "font-weight:bold;font-size:13px">
    <u>parameters</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        auth_token &ensp;: &ensp;turbulence user authorization token.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        dataset_title &ensp;: &ensp;name of the turbulence dataset.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        output_path &ensp;: &ensp;folder path where output files will be to saved to.
        <br>
        &emsp;- n.b. &ensp;: &ensp;cannot be left blank.
    </li>
</ul>

<hr style = "height:6px;border:none;background-color:sienna">

<hr style = "height:6px;border:none;background-color:sienna">

The default token is limited to 4096 points per query. To increase the query points limit to 2 million points, request a free token at: [turbulence.idies.jhu.edu](https://turbulence.idies.jhu.edu/database)

<hr style = "height:6px;border:none;background-color:sienna">

In [8]:
"""
instantiate dataset
"""
from givernylocal.turbulence_dataset import *
from givernylocal.turbulence_toolkit import *
import numpy as np

auth_token = 'edu.jhu.pha.turbulence.testing-201406'
dataset_title = 'isotropic1024coarse'
output_path = '.'

# instantiate the dataset.
dataset = turb_dataset(dataset_title = dataset_title, output_path = output_path, auth_token = auth_token)

<hr style = "height:6px;border:none;background-color:sienna">

<h4 style = "font-weight:bold;font-style:italic">
    getData
</h4>

<p style = "font-weight:bold;font-size:13px">
    <u>purpose</u> : 
    <br>
    &emsp;- retrieve (interpolate and/or differentiate) a group of sparse data points.
    <br>
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>steps</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        step 1 &ensp;: &ensp;identify the database files to be read.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        step 2 &ensp;: &ensp;read the database files and store the interpolated points in an array.
    </li>
</ul>

<p style = "font-weight:bold;font-size:13px">
    <u>parameters</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
         dataset &ensp;: &ensp;the instantiated dataset.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        points &ensp;: &ensp;array of points in the dataset domain.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        variable &ensp;: &ensp;type of data.
        <br>
        &emsp;- vectors &ensp;: &ensp;velocity, vectorpotential, magneticfield, force, position.
        <br>
        &emsp;- scalars &ensp;: &ensp;pressure, temperature, soiltemperature, sgsenergy, sgsviscosity, density.
        <br>
    </li>
    <li style = "font-weight:bold;font-size:13px">
        time &ensp;: &ensp;time (snapshot number for datasets without a full time evolution).
    </li>
    <li style = "font-weight:bold;font-size:13px">
        time_end &ensp;: &ensp;ending time for 'position' variable and time series queries.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        delta_t &ensp;: &ensp;time step for 'position' variable and time series queries.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        temporal_method &ensp;: &ensp;temporal interpolation methods.
        <br>
        &emsp;- none &ensp;: &ensp;No temporal interpolation (the value at the closest stored time will be returned).
        <br>
        &emsp;- pchip &ensp;: &ensp;Piecewise Cubic Hermite Interpolation Polynomial method is used, in which the value from the two nearest times<br>
        &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&nbsp;is interpolated at time t using Cubic Hermite Interpolation Polynomial, with centered finite difference evaluation of the<br>
        &emsp;&emsp;&emsp;&emsp;&emsp;&emsp;&nbsp;end-point time derivatives (i.e. a total of four temporal points are used).
        <br>
    </li>
    <li style = "font-weight:bold;font-size:13px">
        spatial_method &ensp;: &ensp;spatial interpolation and differentiation methods.
        <br>
        &emsp;- none &ensp;: &ensp;No spatial interpolation (value at the datapoint closest to each coordinate value).
        <br>
        &emsp;- lag4 &ensp;: &ensp;4th-order Lagrange Polynomial interpolation along each spatial direction.
        <br>
        &emsp;- lag6 &ensp;: &ensp;6th-order Lagrange Polynomial interpolation along each spatial direction.
        <br>
        &emsp;- lag8 &ensp;: &ensp;8th-order Lagrange Polynomial interpolation along each spatial direction.
        <br>
        &emsp;- m1q4 &ensp;: &ensp;Splines with smoothness 1 (3rd order) over 4 data points.
        <br>
        &emsp;- m2q8 &ensp;: &ensp;Splines with smoothness 2 (5th order) over 8 data points. 
        <br>
        &emsp;- m2q14 &ensp;: &ensp;Splines with smoothness 2 (5th order) over 14 data points. 
        <br>
        &emsp;- fd4noint &ensp;: &ensp;4th-order centered finite differencing (without spatial interpolation).
        <br>
        &emsp;- fd6noint &ensp;: &ensp;6th-order centered finite differencing (without spatial interpolation).
        <br>
        &emsp;- fd8noint &ensp;: &ensp;8th-order centered finite differencing (without spatial interpolation).
        <br>
        &emsp;- fd4lag4 &ensp;: &ensp;4th-order Lagrange Polynomial interpolation in each direction, of the 4th-order finite difference values on the grid.
        <br>
    </li>
    <li style = "font-weight:bold;font-size:13px">
        spatial_operator &ensp;: &ensp;spatial interpolation and differentiation operator.
        <br>
        &emsp;- field &ensp;: &ensp;function evaluation &amp; interpolation.
        <br>
        &emsp;- gradient &ensp;: &ensp;differentiation &amp; interpolation.
        <br>
        &emsp;- hessian &ensp;: &ensp;differentiation &amp; interpolation.
        <br>
        &emsp;- laplacian &ensp;: &ensp;differentiation &amp; interpolation.
        <br>
    </li>
</ul>

<p style = "font-weight:bold;font-size:13px">
    <u>output</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        result &ensp;: &ensp;interpolated/differentiated values array.
    </li>
</ul>

<hr style = "height:6px;border:none;background-color:sienna">

### Define Query range and fields

In [9]:
"""
initialize getData parameters (except time and points)
"""
# variable = 'velocity'
temporal_method = 'none'
spatial_method = 'none'
spatial_operator = 'field'

In [10]:
"""
3D box demo points : evenly spaced over a 3D volume
    - time : the time to be queried (snapshot number for datasets without a full time evolution).
    - nx,ny,nz : number of points along each axis. total number of points queried will be n_points= nx * ny * nz.
    - x_points, y_points, z_points : point distributions along each axis, evenly spaced over the specified ranges.
        - np.linspace(axis minimum, axis maximum, number of points).
    - points : the points array evenly spaced out over the 3D volume.
        - points array is instantiated as an empty array that will be filled inside the for loops.
"""
time = 0.0  # Matches 1 in getCutout. This is physical time

domain_size = 2 * np.pi

# Physical Domain size. See https://turbulence.idies.jhu.edu/docs/isotropic/README-isotropic.pdf
dx = domain_size / 1024

# Physical Time between each snapshot. Can be found in the README file of each dataset
#    e.g. https://turbulence.idies.jhu.edu/docs/isotropic/README-isotropic.pdf
dt = .002

nx = ny = nz = 16

# Evenly sampled points in Physical space
x_points = np.linspace(0 * dx, (nx-1) * dx, nx, dtype=np.float64)
y_points = np.linspace(0 * dx, (nx-1) * dx, ny, dtype=np.float64)
z_points = np.linspace(0 * dx, (nx-1) * dx, nz, dtype=np.float64)

points = np.array(
    [axis.ravel() for axis in np.meshgrid(x_points, y_points, z_points, indexing = 'ij')],
      dtype = np.float64).T

### Call `getData`

In [12]:
"""
use the tools and processing gizmos. May take 1-2 mins
"""
# process interpolation/differentiation of points.
pressure_result = getData(dataset, 'pressure', time, temporal_method, spatial_method, spatial_operator, points)
pressure_3d = np.array(pressure_result).reshape(nx, ny, nz, 1)

velocity_result = getData(dataset, 'velocity', time, temporal_method, spatial_method, spatial_operator, points)
velocity_3d = np.array(velocity_result).reshape(nx, ny, nz, 3)

velocity_3d.shape


-----
getData is processing...

total time elapsed = 0.995 seconds (0.017 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 0.620 seconds (0.010 minutes)

query completed successfully.
-----


(16, 16, 16, 3)

<hr style = "height:6px;border:none;background-color:sienna">

<h4 style = "font-weight:bold;font-style:italic">
    Making Subsequent calls to JHTDB to retrieve more data
</h4>

<p style = "font-weight:bold;font-size:13px">
    <u>purpose</u> : 
    <br>
    &emsp;- A personal token is limited to 2 million points (and the testing token to 4096). In the likely case you need to access more data than that, make SERIAL/SEQUENTIAL calls to JHTDB as follows
    <br>
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>important!</u> : 
    <br>
    &emsp;- Please refrain from making multiple simultaneous tokens! We serve many users, and to maintain reliability, we ask you make 1 query at a time. Note that repeated use of simultaneous queries may get your token access revoked!
    <br>
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>parameters</u> : same as getData
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>output</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        3D array &ensp;: &ensp;same as getData
    </li>
</ul>

<hr style = "height:6px;border:none;background-color:sienna">

In [19]:
# Example: Getting first 10 simulation Timesteps

desired_timesteps = 10
t_start = 0

pressure_result_t10 = []
for i in range(desired_timesteps):
    # Query needs to be made using DNS simulation time
    # In iso1024coarse, dt = 0.002, so the first 10 timesteps are [0, 0.002, 0.004, ..., 0.018]
    t = t_start + i * dt

    pressure_result = getData(dataset, 'pressure', t, temporal_method, spatial_method, spatial_operator, points)
    pressure_3d = np.array(pressure_result).reshape(nx, ny, nz, 1)

    pressure_result_t10.append(pressure_3d)

pressure_result_t10 = np.array(pressure_result_t10)

print("Shape of data is: ", pressure_result_t10.shape, ". 1st dimension is Time")


-----
getData is processing...

total time elapsed = 7.307 seconds (0.122 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 0.887 seconds (0.015 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 0.710 seconds (0.012 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 0.922 seconds (0.015 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 0.610 seconds (0.010 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 0.919 seconds (0.015 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 1.223 seconds (0.020 minutes)

query completed successfully.
-----

-----
getData is processing...

total time elapsed = 1.613 seconds (0.027 minutes)

query completed successfully.
-----

-----
getData is processing...


<hr style = "height:6px;border:none;background-color:sienna">

<h4 style = "font-weight:bold;font-style:italic">
    (Optional) transpose the axes of the retrieved data to match the HuggingFace .h5
</h4>

<p style = "font-weight:bold;font-size:13px">
    <u>purpose</u> : 
    <br>
    &emsp;- The getCutout method used to retrieve the HuggingFace data returns data in [z, y, x] order, whereas the getData() used above uses [x, y, z]. The following cell transposes the getData axes to match
    <br>
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>parameters</u> : none
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>output</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        3D array &ensp;: &ensp;Same dimensions as your getData query result, with x and z axes swapped
    </li>
</ul>

<hr style = "height:6px;border:none;background-color:sienna">

In [26]:
type(pressure_result_t10)

numpy.ndarray

In [28]:
velocity_3d_t = velocity_3d.transpose(2,1,0,3)  # 4th dimension - the variable "depth" should stay where it is
pressure_3d_t = pressure_3d.transpose(2,1,0,3)

pressure_result_t10 = pressure_result_t10.transpose(0, 3,2,1,4)

<hr style = "height:6px;border:none;background-color:sienna">

<h4 style = "font-weight:bold;font-style:italic">
    save interpolation results
</h4>

<p style = "font-weight:bold;font-size:13px">
    <u>purpose</u> : 
    <br>
    &emsp;- save the interpolated/differentiated points retrieved by the getData function.
    <br>
</p>

<p style = "font-weight:bold;font-size:13px">
    <u>parameters</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
         dataset &ensp;: &ensp;the instantiated dataset.
    </li>
    <li style = "font-weight:bold;font-size:13px">
         points &ensp;: &ensp;input points to getData.
    </li>
    <li style = "font-weight:bold;font-size:13px">
         result &ensp;: &ensp;output from getData.
    </li>
    <li style = "font-weight:bold;font-size:13px">
        output_filename &ensp;: &ensp;filename for the tsv file to be saved in the output_path folder.
    </li>
</ul>

<p style = "font-weight:bold;font-size:13px">
    <u>output</u> :
</p>

<ul>
    <li style = "font-weight:bold;font-size:13px">
        tsv file &ensp;: &ensp;saved to output_filename in the output_path folder.
    </li>
</ul>

<hr style = "height:6px;border:none;background-color:sienna">

In [31]:
"""
write the output as a  HDF5 file.
"""
output_filename = 'turbulence-interpolation'


import xarray as xr

# Create a dictionary to hold all variables
data_vars = {}

# Add pressure data for each time step
for t in range(desired_timesteps):
    data_vars[f'Pressure_{t+1:04d}'] = (['x', 'y', 'z', 'phony_dim_3'], pressure_result_t10[t])

# Add coordinate arrays
data_vars['xcoor'] = x_points
data_vars['ycoor'] = y_points
data_vars['zcoor'] = z_points

# Create the dataset
ds = xr.Dataset(
    data_vars=data_vars,
    coords={},
    attrs={
        'dataset': 'isotropic1024coarse',
        't_start': 1,
        't_end': 10,
        't_step': 1,
        'x_start': 1,
        'y_start': 1,
        'z_start': 1,
        'x_end': 16,
        'y_end': 16,
        'z_end': 16,
        'x_step': 1,
        'y_step': 1,
        'z_step': 1,
        'filterWidth': 1
    }
)

ds.to_netcdf('pressure-test.h5', engine='h5netcdf')